# 03 — Average True Range (ATR)

## Goal
Compute ATR(14) — a **volatility ruler** that makes every threshold adaptive to the current market conditions.  
Instead of hardcoding "2 points is a big move", we say "1.5× ATR is a big move" — so the same algorithm works whether ATR = 0.5 or ATR = 5.0.

## Equations

**True Range (TR)** — the largest of three distances:
$$\text{TR}_i = \max\!\Bigl(\text{high}_i - \text{low}_i,\;\bigl|\text{high}_i - \text{close}_{i-1}\bigr|,\;\bigl|\text{low}_i - \text{close}_{i-1}\bigr|\Bigr)$$

The three terms capture:
1. The candle's own span
2. A gap-up (today's high vs yesterday's close)
3. A gap-down (today's low vs yesterday's close)

**ATR — Wilder's Smoothed Moving Average:**
$$\text{ATR}_i = \frac{\text{ATR}_{i-1} \times (n-1) + \text{TR}_i}{n}, \quad n = 14$$

This is an **exponential decay** formula — older candles contribute less, but they never fully disappear.

## Why ATR? (The Ruler Analogy)
Every threshold in this series is expressed as a **multiple of ATR**, not as a fixed price distance.

| Usage | Formula | Where used |
|-------|---------|-----------|
| Base tightness | $\dfrac{\text{base\_width}}{\text{avg\_ATR}} \leq 2.5$ | NB04 |
| Leg strength | $\dfrac{|\text{leg\_net}|}{\text{avg\_ATR}} \geq 1.5$ | NB05 |
| Departure strength | $\dfrac{\text{departure}}{\text{avg\_ATR}} \geq 1.5$ | NB06 |

A 3-point move means nothing by itself. Against ATR = 0.5 that's a 6× move (huge). Against ATR = 5.0 that's only 0.6× (noise). The ATR turns every absolute number into a **relative judgment**.

## Visualization
The upper panel shows price; the lower panel shows the ATR(14) line.  
Notice ATR rises during fast-moving scenarios (e.g. Scenario D) and stays low during tight bases (e.g. Scenario A).


## 1. Load the enriched dataset
Start from the file NB02 wrote (`fixtures_enhanced.csv`) — that already has `range`, `body`, `body-ratio`, `is_base`, `is_doji`, `is_bullish`.

We'll add `atr` to it and save it back so NB04+ can use it.


In [7]:
import pandas as pd
import numpy as np

df = pd.read_csv("../fixtures_enhanced.csv", index_col=0, parse_dates=True)
print(f"Loaded {len(df)} candles")
df.head()


Loaded 98 candles


,open,high,low,close,volume,range,body,body-ratio,is_base,is_doji,is_bullish
Date,,,,,,,,,,,
2024-01-01 00:00:00+00:00,100.0,101.1,99.5,100.6,1000000,1.6,0.6,0.375000,True,False,True
2024-01-08 00:00:00+00:00,100.6,101.1,99.6,100.1,1000000,1.5,0.5,0.333333,True,False,False
2024-01-15 00:00:00+00:00,100.1,101.2,99.6,100.7,1000000,1.6,0.6,0.375000,True,False,True
2024-01-22 00:00:00+00:00,100.7,101.2,99.7,100.2,1000000,1.5,0.5,0.333333,True,False,False
2024-01-29 00:00:00+00:00,100.2,101.3,99.7,100.8,1000000,1.6,0.6,0.375000,True,False,True


## 2. Step one — `prev_close` (yesterday's close)
True Range needs **yesterday's close** to detect overnight gaps. `Series.shift(1)` slides the close column down by one row, so each row now sees the previous row's close.

The very first row's `prev_close` is `NaN` — that's expected; we'll handle it in the next step.


In [8]:
df["prev_close"] = df["close"].shift(1)

df[["close", "prev_close"]].head()


,close,prev_close
Date,,
2024-01-01 00:00:00+00:00,100.6,NaN
2024-01-08 00:00:00+00:00,100.1,100.6
2024-01-15 00:00:00+00:00,100.7,100.1
2024-01-22 00:00:00+00:00,100.2,100.7
2024-01-29 00:00:00+00:00,100.8,100.2


## 3. Compute True Range
For each candle, True Range is the **maximum** of three distances:

1. **Today's range** — `high − low` (the candle's own span)
2. **Gap-up** — `|high − prev_close|` (today shot up above yesterday's close)
3. **Gap-down** — `|low − prev_close|` (today dropped below yesterday's close)

We use vectorized numpy operations rather than a Python loop — much faster and cleaner.

For the first row, `prev_close` is `NaN`, so terms 2 and 3 are `NaN`. We fall back to just term 1 with `fillna(0)` on those two terms before taking the max.


In [9]:
term1 = df["high"] - df["low"]
term2 = (df["high"] - df["prev_close"]).abs().fillna(0)
term3 = (df["low"]  - df["prev_close"]).abs().fillna(0)

df["true_range"] = pd.concat([term1, term2, term3], axis=1).max(axis=1)

df[["high", "low", "prev_close", "true_range"]].head(10)


,high,low,prev_close,true_range
Date,,,,
2024-01-01 00:00:00+00:00,101.1,99.5,NaN,1.6
2024-01-08 00:00:00+00:00,101.1,99.6,100.6,1.5
2024-01-15 00:00:00+00:00,101.2,99.6,100.1,1.6
2024-01-22 00:00:00+00:00,101.2,99.7,100.7,1.5
2024-01-29 00:00:00+00:00,101.3,99.7,100.2,1.6
2024-02-05 00:00:00+00:00,101.3,99.8,100.8,1.5
2024-02-12 00:00:00+00:00,101.4,99.8,100.3,1.6
2024-02-19 00:00:00+00:00,101.4,99.9,100.9,1.5
2024-02-26 00:00:00+00:00,101.5,99.9,100.4,1.6


## 4. Smooth True Range into ATR (Wilder's method)
Raw True Range is noisy — a single big candle would spike it. Wilder's smoothing averages it out over 14 periods using this recurrence:

$$\text{ATR}_i = \frac{\text{ATR}_{i-1} \times (n-1) + \text{TR}_i}{n}, \quad n = 14$$

This is mathematically equivalent to an EMA with $\alpha = 1/n$ — older values decay exponentially but never disappear entirely.

**Seeding the recurrence**: the very first ATR value is just `TR[0]` (no history to average with). From index 1 onwards the formula kicks in.

We compute this with an explicit loop. Vectorized version exists but the loop matches the formula one-for-one and is easier to read.


In [10]:
n = 14   # standard ATR period

tr  = df["true_range"].to_numpy()
atr = tr.copy()                                    # seed: ATR[0] = TR[0]

for i in range(1, len(tr)):
    atr[i] = (atr[i - 1] * (n - 1) + tr[i]) / n    # Wilder smoothing

df["atr"] = atr

df[["true_range", "atr"]].head(20)


,true_range,atr
Date,,
2024-01-01 00:00:00+00:00,1.6,1.600000
2024-01-08 00:00:00+00:00,1.5,1.592857
2024-01-15 00:00:00+00:00,1.6,1.593367
2024-01-22 00:00:00+00:00,1.5,1.586698
2024-01-29 00:00:00+00:00,1.6,1.587648
2024-02-05 00:00:00+00:00,1.5,1.581388
2024-02-12 00:00:00+00:00,1.6,1.582717
2024-02-19 00:00:00+00:00,1.5,1.576809
2024-02-26 00:00:00+00:00,1.6,1.578465


## 5. Save the enriched dataset (now with ATR)
Drop the intermediate `prev_close` and `true_range` columns — they served their purpose. Keep `atr` and persist back to disk so NB04+ can read it.


In [11]:
df = df.drop(columns=["prev_close", "true_range"], errors="ignore")
df.to_csv("../fixtures_enhanced.csv")
print("Saved (now includes 'atr' column)")
df.head()


Saved (now includes 'atr' column)


,open,high,low,close,volume,range,body,body-ratio,is_base,is_doji,is_bullish,atr
Date,,,,,,,,,,,,
2024-01-01 00:00:00+00:00,100.0,101.1,99.5,100.6,1000000,1.6,0.6,0.375000,True,False,True,1.600000
2024-01-08 00:00:00+00:00,100.6,101.1,99.6,100.1,1000000,1.5,0.5,0.333333,True,False,False,1.592857
2024-01-15 00:00:00+00:00,100.1,101.2,99.6,100.7,1000000,1.6,0.6,0.375000,True,False,True,1.593367
2024-01-22 00:00:00+00:00,100.7,101.2,99.7,100.2,1000000,1.5,0.5,0.333333,True,False,False,1.586698
2024-01-29 00:00:00+00:00,100.2,101.3,99.7,100.8,1000000,1.6,0.6,0.375000,True,False,True,1.587648


## 6. Visualization — see ATR breathe with the market
Two stacked panels:
- **Top** — the candles
- **Bottom** — the ATR(14) line filled to zero, in soft blue

What you should observe:
- ATR is **flat and low** during the warmup and quiet sections
- ATR **climbs** when a volatile scenario hits (large candles spike TR, which feeds into ATR)
- ATR **decays slowly** afterwards — older volatility lingers (that's Wilder's smoothing at work)

This single line is what makes every threshold in NB04–NB07 adaptive instead of hardcoded.


In [12]:
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots

pio.renderers.default = "notebook_connected"

COLOR_BULL = "#26a69a"
COLOR_BEAR = "#ef5350"
COLOR_ATR  = "#7c83fd"
BG         = "#131722"
GRID       = "#1e222d"

fig = make_subplots(
    rows=2, cols=1,
    shared_xaxes=True,
    row_heights=[0.65, 0.35],
    vertical_spacing=0.03,
)

# top — candles
fig.add_trace(go.Candlestick(
    x=df.index,
    open=df["open"], high=df["high"],
    low=df["low"],   close=df["close"],
    increasing_line_color=COLOR_BULL,
    decreasing_line_color=COLOR_BEAR,
    name="Price",
), row=1, col=1)

# bottom — ATR line (filled to zero for visual weight)
fig.add_trace(go.Scatter(
    x=df.index,
    y=df["atr"],
    mode="lines",
    line=dict(color=COLOR_ATR, width=1.5),
    fill="tozeroy",
    fillcolor="rgba(124,131,253,0.10)",
    name=f"ATR({n})",
    hovertemplate="%{x}<br>ATR: %{y:.3f}<extra></extra>",
), row=2, col=1)

# dark theme
fig.update_layout(
    title=f"ATR({n}) — Average True Range",
    height=580,
    plot_bgcolor=BG, paper_bgcolor=BG,
    font_color="white",
    xaxis_rangeslider_visible=False,
    hovermode="x unified",
    legend=dict(orientation="h", y=1.04),
)
shared_axis = dict(gridcolor=GRID, showgrid=True, zeroline=False, linecolor=GRID)
fig.update_xaxes(**shared_axis)
fig.update_yaxes(**shared_axis)
fig.update_yaxes(title_text="Price",    row=1, col=1)
fig.update_yaxes(title_text=f"ATR({n})", row=2, col=1)

fig.show()
